In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
DATA_ROOT = PROJECT_ROOT / 'data'
RUN_ROOT = PROJECT_ROOT / 'runs'
QASMBENCH_ROOT = DATA_ROOT / 'QASMBench'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RUN_ROOT:", RUN_ROOT)


ModuleNotFoundError: No module named 'google.colab'

: 

In [ ]:
# Same meaningful graphs, but display only (no file saved)

from pathlib import Path
import json, csv, math
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
RUN_NAME = 'ppo_2stages_gridv5_20260325_195948_s42'  # change if needed
STAGE = 'stage2_mid'
RUN_DIR = PROJECT_ROOT / 'runs' / RUN_NAME
STAGE_DIR = RUN_DIR / STAGE
EVAL_DIR = STAGE_DIR / 'eval'

def fnum(x):
    try:
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return None
        return v
    except Exception:
        return None

def get_metric(d, k1, k2):
    v = d.get(k1, None)
    if v is None:
        v = d.get(k2, None)
    return fnum(v)

snapshots = sorted(EVAL_DIR.glob('update_*.json'))
if not snapshots:
    raise FileNotFoundError(f'No eval snapshots in: {EVAL_DIR}')

rows = []
for p in snapshots:
    data = json.loads(p.read_text(encoding='utf-8'))
    upd = int(data.get('update', p.stem.split('_')[-1]))
    per_topo = data.get('per_topology', {}) or {}
    for topo, m in per_topo.items():
        improve = get_metric(m, 'eval_improvement_pct', 'mean_improvement_pct')
        win = get_metric(m, 'eval_win_rate', 'win_rate_vs_sabre')
        timeout = get_metric(m, 'eval_timeout_rate', 'timeout_rate')
        ppo_sw = get_metric(m, 'eval_mean_ppo_swaps', 'mean_model_swaps')
        sabre_sw = get_metric(m, 'eval_mean_sabre_swaps', 'mean_sabre_swaps')
        rows.append({
            'update': upd,
            'topology': topo,
            'improve': improve,
            'win': win,
            'timeout': timeout,
            'ppo_swaps': ppo_sw,
            'sabre_swaps': sabre_sw,
            'swap_ratio': (ppo_sw / sabre_sw) if (ppo_sw is not None and sabre_sw not in (None, 0.0)) else None
        })

if not rows:
    raise RuntimeError('No per-topology eval rows found.')

best_update = None
metrics_csv = STAGE_DIR / 'metrics.csv'
if metrics_csv.exists():
    eval_rows = []
    with metrics_csv.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for r in reader:
            imp = fnum(r.get('eval_improvement_pct'))
            upd = fnum(r.get('update'))
            if imp is not None and upd is not None:
                eval_rows.append((int(upd), imp))
    if eval_rows:
        best_update = max(eval_rows, key=lambda t: t[1])[0]

if best_update is None:
    by_update = {}
    for r in rows:
        if r['improve'] is None:
            continue
        by_update.setdefault(r['update'], []).append(r['improve'])
    if by_update:
        best_update = max(by_update.items(), key=lambda kv: np.mean(kv[1]))[0]

topologies = sorted({r['topology'] for r in rows})
updates = sorted({r['update'] for r in rows})
colors = {'linear_5': '#1f77b4', 'grid_3x3': '#d62728', 'heavy_hex_19': '#2ca02c'}

def series_for(topo, key):
    xs, ys = [], []
    for u in updates:
        vals = [r[key] for r in rows if r['topology'] == topo and r['update'] == u and r[key] is not None]
        if vals:
            xs.append(u)
            ys.append(float(np.mean(vals)))
    return xs, ys

fig, axs = plt.subplots(2, 2, figsize=(14, 9))
ax1, ax2, ax3, ax4 = axs.ravel()

# A) Improvement vs SABRE
for topo in topologies:
    xs, ys = series_for(topo, 'improve')
    if xs:
        ax1.plot(xs, ys, marker='o', label=topo, color=colors.get(topo))
ax1.axhline(0.0, color='black', linestyle='--', linewidth=1)
if best_update is not None:
    ax1.axvline(best_update, color='gray', linestyle=':', linewidth=1.5, label=f'best_update={best_update}')
ax1.set_title('Improvement vs SABRE by Topology')
ax1.set_xlabel('Update')
ax1.set_ylabel('Improvement % (higher is better)')
ax1.grid(alpha=0.25)
ax1.legend()

# B) Swap ratio PPO/SABRE
for topo in topologies:
    xs, ys = series_for(topo, 'swap_ratio')
    if xs:
        ax2.plot(xs, ys, marker='o', label=topo, color=colors.get(topo))
ax2.axhline(1.0, color='black', linestyle='--', linewidth=1)
if best_update is not None:
    ax2.axvline(best_update, color='gray', linestyle=':', linewidth=1.5)
ax2.set_title('Swap Ratio (PPO / SABRE)')
ax2.set_xlabel('Update')
ax2.set_ylabel('Ratio (1.0 = equal)')
ax2.grid(alpha=0.25)
ax2.legend()

# C) Grid win/timeout
focus_topo = 'grid_3x3' if 'grid_3x3' in topologies else topologies[0]
xs_w, ys_w = series_for(focus_topo, 'win')
xs_t, ys_t = series_for(focus_topo, 'timeout')
ax3.plot(xs_w, ys_w, marker='o', label=f'{focus_topo} win_rate', color='#9467bd')
ax3.plot(xs_t, ys_t, marker='o', label=f'{focus_topo} timeout_rate', color='#ff7f0e')
if best_update is not None:
    ax3.axvline(best_update, color='gray', linestyle=':', linewidth=1.5)
ax3.set_title(f'{focus_topo}: Win/Timeout Evolution')
ax3.set_xlabel('Update')
ax3.set_ylabel('Rate')
ax3.set_ylim(0, 1.05)
ax3.grid(alpha=0.25)
ax3.legend()

# D) PPO vs SABRE swaps at best update
if best_update is not None:
    bar_topos, ppo_vals, sabre_vals = [], [], []
    for topo in topologies:
        cand = [r for r in rows if r['topology'] == topo and r['update'] == best_update]
        if not cand:
            continue
        ppo_m = np.mean([c['ppo_swaps'] for c in cand if c['ppo_swaps'] is not None]) if any(c['ppo_swaps'] is not None for c in cand) else None
        sab_m = np.mean([c['sabre_swaps'] for c in cand if c['sabre_swaps'] is not None]) if any(c['sabre_swaps'] is not None for c in cand) else None
        if ppo_m is not None and sab_m is not None:
            bar_topos.append(topo)
            ppo_vals.append(ppo_m)
            sabre_vals.append(sab_m)

    x = np.arange(len(bar_topos))
    w = 0.38
    ax4.bar(x - w/2, ppo_vals, width=w, label='PPO', color='#4C78A8')
    ax4.bar(x + w/2, sabre_vals, width=w, label='SABRE', color='#F58518')
    ax4.set_xticks(x)
    ax4.set_xticklabels(bar_topos)
    ax4.set_title(f'Swaps at Best Eval Update ({best_update})')
    ax4.set_ylabel('Mean SWAP count')
    ax4.grid(axis='y', alpha=0.25)
    ax4.legend()
else:
    ax4.text(0.5, 0.5, 'No best_update found', ha='center', va='center')
    ax4.axis('off')

fig.suptitle(f'Run Diagnostics: {RUN_NAME} / {STAGE}', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print("Best update used:", best_update)
